In [2]:
import os
import numpy as np
import pandas as pd
import boto3
import time
import helper
import sys
from openpyxl import Workbook, load_workbook
cwd = os.getcwd()
print(cwd)

/home/sagemaker-user/CAPE_PERFORMANCE


#### Load the matched cape and model data

In [3]:
bucket_name = 'pr-home-datascience'
prefix = 'Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/'

may_keywords = 'processed_combinedv4v5.csv'
# may_data_all = read_data.read_s3(bucket_name, prefix, may_keywords)


In [4]:
model_no_HPPREF_length = 1864631
may_length = 757655
july_length = 2000358 # is same for both V4 and V5

#### Overall distribution, sanity check

In [5]:

def write_match_summary_short(read_data, bucket_name, prefix,
                              model_no_HPPREF_length, may_length, july_length,
                              out_path="match_summary.xlsx"):
    months = ["may", "july"]
    versions = ["v4", "v5"]
    month_len = {"may": may_length, "july": july_length}

    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
        wb = writer.book
        pct_fmt = wb.add_format({"num_format": "0.00%"})
        int_fmt = wb.add_format({"num_format": "0"})
        left_fmt = wb.add_format({"align": "left"})

        for m in months:
            for v in versions:
                fname = f"model_{m}_{v}.csv"
                df = read_data.read_s3(bucket_name, prefix, fname)
                n = len(df)
                pct_model = n / model_no_HPPREF_length
                pct_cape = n / month_len[m]

                summary = pd.DataFrame({
                    "Metric": [
                        "model_w/o_HPPREF samples",
                        f"cape samples in {m}",
                        "matched records",
                        "pct of model_w/o_HPPREF",
                        f"pct of cape in {m}"
                    ],
                    "Value": [
                        model_no_HPPREF_length,
                        month_len[m],
                        n,
                        pct_model,
                        pct_cape
                    ],
                    "File": [fname] + [""] * 4
                })

                sheet = f"{m}_{v}"
                summary.to_excel(writer, index=False, sheet_name=sheet)
                ws = writer.sheets[sheet]
                ws.set_column("A:C", 30, left_fmt)
                ws.set_column("B:B", None, int_fmt)
                ws.write_number(3 + 1, 1, pct_model, pct_fmt)
                ws.write_number(4 + 1, 1, pct_cape, pct_fmt)
                ws.freeze_panes(1, 0)

    print(f"✅ Wrote: {out_path}")

In [5]:
write_match_summary_short(
    read_data, bucket_name, prefix,
    model_no_HPPREF_length, may_length, july_length,
    out_path="./match_summary.xlsx"
)

Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_may_v4.csv
Load file successfully, file length is  616091
Now the total rows are  616091
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_may_v5.csv
Load file successfully, file length is  616091
Now the total rows are  616091
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_july_v4.csv
Load file successfully, file length is  725171
Now the total rows are  725171
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_july_v5.csv
Load file successfully, file length is  725171
Now the total rows are  725171
✅ Wrote: ./match_summary.xlsx


#### Divide into train and val based on 'tv' in model

In [6]:
def split_train_val(read_data, bucket_name, prefix, month, version, key="pol_num"):
    """Split model and cape data into train/val sets with aligned indices."""
    model_file = f"model_{month}_{version}.csv"
    cape_file  = f"cape_{month}_{version}.csv"

    model_df = read_data.read_s3(bucket_name, prefix, model_file)
    cape_df  = read_data.read_s3(bucket_name, prefix, cape_file)

    # Keep original column names for later filtering
    if month == 'may':
        cape_df['cape_run_dt'] = cape_df['effective_date']
    model_cols = model_df.columns.tolist()
    cape_cols  = cape_df.columns.tolist()

    # Ensure same type for key
    model_df[key] = model_df[key].astype(str).str.strip()
    cape_df[key]  = cape_df[key].astype(str).str.strip()

    # Build train/val key lists
    train_keys = model_df.loc[model_df["tv"] == "T", key]
    val_keys   = model_df.loc[model_df["tv"] == "V", key]

    # Filter both model and cape by those keys
    model_train = model_df[model_df[key].isin(train_keys)].reset_index(drop=True)
    model_val   = model_df[model_df[key].isin(val_keys)].reset_index(drop=True)

    cape_train = cape_df[cape_df[key].isin(train_keys)].reset_index(drop=True)
    cape_val   = cape_df[cape_df[key].isin(val_keys)].reset_index(drop=True)


    # Optional: align indices for direct comparison
    model_train.index = cape_train.index
    model_val.index   = cape_val.index

    # ✨ Ensure only original variables are kept
    model_train = model_train[model_cols]
    model_val   = model_val[model_cols]
    cape_train  = cape_train[cape_cols]
    cape_val    = cape_val[cape_cols]

    print(f"✅ {month}_{version}:")
    print(f"  Model train: {len(model_train)}, val: {len(model_val)}")
    print(f"  Cape  train: {len(cape_train)},  val: {len(cape_val)}")

    return model_train, model_val, cape_train, cape_val

def tag_and_collect(df, month, version, split, src, keep_cols):
    out = df[keep_cols].copy()
    # prepend minimal metadata columns (these are the ONLY non-original cols)
    out.insert(0, "src", src)
    out.insert(1, "split", split)
    out.insert(2, "month", month)
    out.insert(3, "version", version)
    return out

In [12]:
model_train, model_val, cape_train, cape_val = split_train_val(
            read_data, bucket_name, prefix, month='may', version='v4')
keywords = 'tv'

key_columns = [col for col in cape_train.columns if keywords in col.lower()]
print('There are ', len(key_columns), ' variables contains ' + keywords + '.')
print('They are ', key_columns)

Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_may_v4.csv
Load file successfully, file length is  616091
Now the total rows are  616091
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/cape_may_v4.csv
Load file successfully, file length is  616091
Now the total rows are  616091
✅ may_v4:
  Model train: 432052, val: 184039
  Cape  train: 432052,  val: 184039
There are  2  variables contains dt.
They are  ['orgl_pol_eff_dt', 'cape_run_dt']


Add the statistical results for train and valid, see whether reasonable.

In [13]:

def add_information_to_excel(model_train, model_val, cape_train, cape_val,
                             out_path, m, v, sheet_name="summary"):
    """Append model/cape train-val info to the bottom of an existing Excel sheet."""
    # ---- Compute stats ----
    mt, mv = len(model_train), len(model_val)
    ct, cv = len(cape_train), len(cape_val)
    tm, tc = mt + mv, ct + cv

    row = pd.DataFrame([{
        "month_version": f"{m}_{v}",
        "model_train": mt, "model_val": mv, "total_model": tm,
        "cape_train": ct, "cape_val": cv, "total_cape": tc,
    }])
    row_per = pd.DataFrame([{
        'sep1': np.nan,
        "pct_model_train": f"{(mt/tm):.2%}" if tm else "0.00%",
        "pct_model_val":   f"{(mv/tm):.2%}" if tm else "0.00%",
        'sep2': np.nan, 
        "pct_cape_train":  f"{(ct/tc):.2%}" if tc else "0.00%",
        "pct_cape_val":    f"{(cv/tc):.2%}" if tc else "0.00%",
    }])
    row_T = row.T.reset_index()
    row_T.columns = ["Variable", "Value"]
    row_per_T = row_per.T.reset_index()
    row_per_T.columns = ["Metric", "Value"]
    row_per_T = row_per_T[["Value"]]
    try:
        book = load_workbook(out_path)
        ws = book[sheet_name]
        start_row = ws.max_row  # 1-based
        with pd.ExcelWriter(out_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
            row_T.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=start_row)
            row_per_T.to_excel(writer, sheet_name=sheet_name, index=False, header=False,startcol=2, startrow=start_row)
        print(f"✅ Appended {m}_{v} to {sheet_name} in {out_path}")
    except:
        print('No file found')

In [14]:
all_chunks = []
months = ['may', 'july']
versions = ['v4', 'v5']


for m in months:
    for v in versions:
        model_train, model_val, cape_train, cape_val = split_train_val(
            read_data, bucket_name, prefix, month=m, version=v
        )
        # 2) Capture each source’s original columns ONCE
        if v == 'v4':
            MODEL_COLS_v4 = model_train.columns.tolist()
            CAPE_COLS_v4  = cape_train.columns.tolist()
        elif v == 'v5':
            MODEL_COLS_v5 = model_train.columns.tolist()
            CAPE_COLS_v5  = cape_train.columns.tolist()
        MODEL_COLS = model_train.columns.tolist()
        CAPE_COLS  = cape_train.columns.tolist()        

        add_information_to_excel(model_train, model_val, cape_train, cape_val,
                                 os.getcwd()+"/match_summary.xlsx", m, v, sheet_name=m+'_'+v)

        for df, split, src, keep_cols in [
            (model_train, "train", "model", MODEL_COLS),
            (model_val,   "val",   "model", MODEL_COLS),
            (cape_train,  "train", "cape",  CAPE_COLS),
            (cape_val,    "val",   "cape",  CAPE_COLS),
        ]:
            all_chunks.append(tag_and_collect(df, m, v, split, src, keep_cols))

combined_all = pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame()



Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_may_v4.csv
Load file successfully, file length is  616091
Now the total rows are  616091
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/cape_may_v4.csv
Load file successfully, file length is  616091
Now the total rows are  616091
✅ may_v4:
  Model train: 432052, val: 184039
  Cape  train: 432052,  val: 184039
✅ Appended may_v4 to may_v4 in /home/sagemaker-user/CAPE_V4_V5_PERFORMANCE/match_summary.xlsx
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_may_v5.csv
Load file successfully, file length is  616091
Now the total rows are  616091
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/cape_may_v5.csv
Load file successfully, file length is  616091
Now the total rows are  61

In [22]:
print('effetive_date' in CAPE_COLS_v4)
print('cape_run_dt' in CAPE_COLS_v4)

False
True


In [ ]:
print('effetive_date' in CAPE_COLS_v4)

False


In [23]:
# model_train.head(10)
cape_train.head(10)

,cape_response_status,cape_response_description,cape_response_id,cape_primary_structure_latitude,cape_primary_structure_longitude,attribute_geometry_id,cape_parcel_id,cape_oblique_property_date,cape_oblique_property_image_source,cape_oblique_property_image_url,...,cape_roof_structural_damage_end_date,cape_roof_structural_damage_image_source,cape_roof_structural_damage_image_url,cape_roof_tarp,cape_roof_tarp_confidence,cape_roof_tarp_start_date,cape_roof_tarp_end_date,cape_roof_tarp_image_source,cape_roof_tarp_image_url,cape_attribution_url
0,OUTSIDE_COVERAGE_AREA,The input location is outside of Cape's curren...,0a6b0dd6-b35d-4f79-8858-f5a1bcc1be75,41.619515,-70.936559,NaN,550112679.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://flow.capeanalytics.com/api/v1/attribut...
1,OUTSIDE_COVERAGE_AREA,The input location is outside of Cape's curren...,de882c1f-01c6-4632-9733-dbcebde063ca,41.619515,-70.936559,NaN,550112679.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://flow.capeanalytics.com/api/v1/attribut...
2,OK,API request was successful,95cbc291-7b13-440e-9551-9614717351f6,41.619578,-70.936620,4.665763e+09,550112679.0,NaN,NaN,NaN,...,2015-06-10,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/95cb...,no_roof_tarp,0.9964,NaN,2015-06-10,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/95cb...,https://flow.capeanalytics.com/api/v1/attribut...
3,OK,API request was successful,857e48c5-798b-4965-95fc-81460d06ea4d,41.619578,-70.936620,4.665763e+09,550112679.0,NaN,NaN,NaN,...,2015-06-10,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/857e...,no_roof_tarp,0.9964,NaN,2015-06-10,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/857e...,https://flow.capeanalytics.com/api/v1/attribut...
4,OK,API request was successful,3c9c3ce8-c0a2-4ff5-a818-69dd2acb6f0d,41.619570,-70.936611,3.668511e+09,550112679.0,NaN,NaN,NaN,...,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,no_roof_tarp,0.9967,2015-06-10,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,https://flow.capeanalytics.com/api/v1/attribut...
5,OK,API request was successful,3c9c3ce8-c0a2-4ff5-a818-69dd2acb6f0d,41.619570,-70.936611,3.668511e+09,550112679.0,NaN,NaN,NaN,...,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,no_roof_tarp,0.9967,2015-06-10,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,https://flow.capeanalytics.com/api/v1/attribut...
6,OK,API request was successful,3c9c3ce8-c0a2-4ff5-a818-69dd2acb6f0d,41.619570,-70.936611,3.668511e+09,550112679.0,NaN,NaN,NaN,...,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,no_roof_tarp,0.9967,2015-06-10,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,https://flow.capeanalytics.com/api/v1/attribut...
7,OK,API request was successful,3c9c3ce8-c0a2-4ff5-a818-69dd2acb6f0d,41.619570,-70.936611,3.668511e+09,550112679.0,NaN,NaN,NaN,...,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,no_roof_tarp,0.9967,2015-06-10,2017-05-16,aerial_high_res,https://flow.capeanalytics.com/api/v3/img/3c9c...,https://flow.capeanalytics.com/api/v1/attribut...
8,OUTSIDE_COVERAGE_AREA,The input location is outside of Cape's curren...,06b31742-e8bc-43ad-9b54-0b7e020e96f3,41.670828,-70.929522,NaN,548494880.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://flow.capeanalytics.com/api/v1/attribut...
9,OUTSIDE_COVERAGE_AREA,The input location is outside of Cape's curren...,7373920c-ea5d-4d22-b562-25e1abb1ca36,41.670828,-70.929522,NaN,548494880.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://flow.capeanalytics.com/api/v1/attribut...


Save the data into different files, easy for future check.

In [24]:
output_path = 's3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/'

for v in versions:
    dfv = combined_all[combined_all["version"] == v]
    for split in ["train", "val"]:
        for src in ["model", "cape"]:
            dfs = dfv[(dfv["split"] == split) & (dfv["src"] == src)]
            if not dfs.empty:
                # different source has different column names
                if v == 'v4':
                    if src == "model":
                        cols_to_save = [c for c in MODEL_COLS_v4 if c in dfs.columns]
                    else:
                        cols_to_save = [c for c in CAPE_COLS_v4 if c in dfs.columns]
                if v == 'v5':
                    if src == "model":
                        cols_to_save = [c for c in MODEL_COLS_v5 if c in dfs.columns]
                    else:
                        cols_to_save = [c for c in CAPE_COLS_v5 if c in dfs.columns]
                # output path
                out_path = f"{output_path}{src}_{split}_{v}.csv"
                dfs[cols_to_save].to_csv(out_path, index=False)

                print(f"✅ Saved: {out_path}  (rows={len(dfs)}, cols={len(cols_to_save)})")
                


✅ Saved: s3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/model_train_v4.csv  (rows=940672, cols=15)
✅ Saved: s3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/cape_train_v4.csv  (rows=940672, cols=103)
✅ Saved: s3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/model_val_v4.csv  (rows=400586, cols=15)
✅ Saved: s3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/cape_val_v4.csv  (rows=400586, cols=103)
✅ Saved: s3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/model_train_v5.csv  (rows=940672, cols=15)
✅ Saved: s3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/cape_train_v5.csv  (rows=940672, cols=118)
✅ Saved: s3://pr-hom

In [14]:
target_keys = ['CTH00002032331', 'MAH00002059633', 'NJH00002100606', 'NJH00002102101']


key = 'pol_num'
for m in months[1:2]:
    for v in versions[0:1]:
        model_file = f"model_{m}_{v}.csv"
        model_df = read_data.read_s3(bucket_name, prefix, model_file)
        for target_key in target_keys:
            model_rows = model_df[model_df[key] == target_key]

            print("\nModel rows:")
            print(model_rows)



SyntaxError: invalid syntax (1130658810.py, line 1)